# LightGBM — Housing Rent Prediction

Trains **LightGBM** on the identical prepared data used by every model, evaluates it on the training split (5-fold CV), validation and test, and writes results to `reports/`.

- **Configuration:** tuned hyperparameters (5-fold CV search on train (GridSearchCV for linear, RandomizedSearchCV for trees/boosting)); defined once in `src/models.py → FINAL_PARAMS`.
- **Target:** `Rent`, modelled on the log1p scale; all metrics are in rupees.
- **Scaling:** none (tree-based or constant model).
- **Split:** fixed 70/15/15, `random_state=42`.

In [ ]:
# Needs lightgbm (macOS: brew install libomp)
# !pip install lightgbm

In [ ]:
import sys, pathlib, warnings
sys.path.insert(0, str(pathlib.Path.cwd().parent))   # repo root -> `import src`
warnings.filterwarnings('ignore')
from src.preprocessing import prepare_data, assert_run_consistency
from src.models import make_model, display_name, final_params, PARAM_SOURCE
from src.evaluation import evaluate, results_table, print_summary, save_results, plot_diagnostics

In [ ]:
MODEL_KEY = 'lightgbm_model'
EVALUATE_TEST = True   # final evaluation stage

In [ ]:
# Identical data preparation for every model (src pipeline, deterministic by seed)
data = prepare_data()
ref = assert_run_consistency(data)
n_rows = sum(data['meta']['rows'][s] for s in ('train', 'val', 'test'))
print(f"{data['X_train'].shape[0]} train / {data['X_val'].shape[0]} val / {data['X_test'].shape[0]} test | "
      f"{len(data['feature_names'])} features | reference_id {ref}")

### Model and its final hyperparameters

In [ ]:
print(display_name(MODEL_KEY), '|', PARAM_SOURCE[MODEL_KEY])
print(final_params(MODEL_KEY) or 'library defaults')
model = make_model(MODEL_KEY, data['target_transform'])
model

### Train, cross-validate, evaluate

In [ ]:
results, pred, _ = evaluate(model, data, display_name(MODEL_KEY), MODEL_KEY, evaluate_test=EVALUATE_TEST)
print_summary(results, n_rows, len(data['feature_names']))
save_results(results, pred, data)   # -> reports/metrics, reports/predictions
results_table(results)

### Diagnostics

In [ ]:
plot_diagnostics(results, pred, data, model, display_name(MODEL_KEY));